# Module 2: From Probability to Prediction

In the previous module, probability distributions helped us describe uncertain outcomes.

In practice, however, we rarely have access to every customer, transaction, employee, patient, or product that belongs to the population we want to study.

Instead, we usually work with a sample.

This creates an important question:

> How can information from a sample be used to learn about a much larger population?

This module develops that idea through:

- Population and Sample
- Sampling
- Sampling Distribution
- Central Limit Theorem
- Standard Error
- Point Estimation
- Interval Estimation
- Confidence Intervals
- Confidence Level
- Margin of Error

The main journey is:

$$
\text{Population}
\rightarrow
\text{Sample}
\rightarrow
\text{Sample Statistic}
\rightarrow
\text{Estimate Population Parameter}
$$

## 1. Population and Sample

Suppose an e-commerce company has 100,000 customers.

The company wants to know the average amount spent by its customers.

The complete set of 100,000 customers is the **population**.

A smaller group selected from these customers is a **sample**.

$$
\text{Population}
=
\text{Complete group we want to understand}
$$

$$
\text{Sample}
=
\text{Subset of the population that we actually observe}
$$

The goal of inferential statistics is to use the sample to make reasonable conclusions about the population.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# Simulated spending data for 100,000 customers
population = np.random.gamma(shape=2.5, scale=800, size=100000)

# Randomly select 100 customers
sample = np.random.choice(population, size=100, replace=False)

print("Population size:", len(population))
print("Sample size:", len(sample))

print("\nPopulation mean:", round(population.mean(), 2))
print("Sample mean:", round(sample.mean(), 2))

### What do we notice?

The sample mean is unlikely to be exactly equal to the population mean.

This is normal.

Different random samples contain different observations, so they produce slightly different results.

This difference between a sample statistic and the true population parameter is part of **sampling variability**.

In real-world analysis, the population mean is usually unknown. We calculate the sample mean because it provides information about the population mean.

## 2. Parameter and Statistic

A **parameter** describes a population.

A **statistic** describes a sample.

Common examples are:

| Population Parameter | Sample Statistic |
|---|---|
| Population Mean $\mu$ | Sample Mean $\bar{x}$ |
| Population Standard Deviation $\sigma$ | Sample Standard Deviation $s$ |
| Population Proportion $p$ | Sample Proportion $\hat{p}$ |

For example:

$$
\mu = \text{True average spending of all customers}
$$

while:

$$
\bar{x} = \text{Average spending observed in our sample}
$$

The population parameter is what we want to know.

The sample statistic is what we can calculate.

## 3. Why Do We Use Samples?

Using the entire population may be:

- Expensive
- Time-consuming
- Operationally difficult
- Sometimes impossible

Consider a few examples.

A company may have millions of customers.

A factory cannot destroy every product to test product durability.

A political survey cannot realistically ask every voter.

A streaming platform cannot manually study the behaviour of every user.

Sampling allows us to study a manageable part of the population and use statistical methods to draw conclusions about the whole.

## 4. Random Sampling

A useful sample should represent the population reasonably well.

One common approach is **random sampling**, where every observation has a fair chance of being selected.

Consider the customer population created earlier.

We can draw several random samples of 100 customers.

In [ ]:
sample_1 = np.random.choice(population, size=100, replace=False)
sample_2 = np.random.choice(population, size=100, replace=False)
sample_3 = np.random.choice(population, size=100, replace=False)

print("Sample 1 Mean:", round(sample_1.mean(), 2))
print("Sample 2 Mean:", round(sample_2.mean(), 2))
print("Sample 3 Mean:", round(sample_3.mean(), 2))

### Observation

Each sample gives us a different sample mean.

We may observe something like:

$$
\bar{x}_1 \neq \bar{x}_2 \neq \bar{x}_3
$$

This leads to an important question:

> If every sample gives a slightly different answer, how much can we trust a sample mean?

To answer this, we need to study the behaviour of sample means themselves.

# 5. Sampling Distribution

Suppose we repeatedly perform the following process:

1. Take a random sample from the population.
2. Calculate its mean.
3. Store the sample mean.
4. Repeat the process many times.

We now have a new distribution.

This is not the distribution of individual customer spending.

It is the distribution of **sample means**.

This is called a **sampling distribution of the sample mean**.

$$
\bar{X}
=
\text{Random variable representing sample means}
$$

This distinction is important:

$$
X
=
\text{Individual observation}
$$

$$
\bar{X}
=
\text{Mean of a sample}
$$

In [ ]:
# Take 1000 samples
# Each sample contains 30 customers

sample_means = []

for i in range(1000):
    current_sample = np.random.choice(population, size=30, replace=True)
    sample_means.append(current_sample.mean())

sample_means = np.array(sample_means)

print("Population Mean:", round(population.mean(), 2))
print("Mean of Sample Means:", round(sample_means.mean(), 2))

### Observation

The individual sample means vary.

However, when many samples are taken, the average of those sample means tends to stay close to the population mean.

Mathematically:

$$
E(\bar{X}) = \mu
$$

This tells us that the sample mean is a useful estimator of the population mean.

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(sample_means, bins=30, edgecolor="black")
plt.axvline(population.mean(), linestyle="--", label="Population Mean")
plt.xlabel("Sample Mean")
plt.ylabel("Frequency")
plt.title("Sampling Distribution of the Sample Mean")
plt.legend()
plt.show()

# 6. Central Limit Theorem

The Central Limit Theorem explains one of the most useful patterns in statistics.

Suppose a population has:

- Mean $\mu$
- Standard deviation $\sigma$

If we repeatedly take sufficiently large random samples of size $n$ and calculate their means, the distribution of those sample means tends to become approximately Normal.

This can happen even when the original population itself is not Normally distributed.

The sampling distribution has approximately:

$$
\mu_{\bar{X}} = \mu
$$

and:

$$
\sigma_{\bar{X}} = \frac{\sigma}{\sqrt{n}}
$$

The second quantity is called the **Standard Error**.

The Central Limit Theorem allows us to use sample statistics to reason about unknown population parameters.

## 7. Seeing the Central Limit Theorem in Action

The customer spending population used in this notebook is intentionally skewed.

Let us first look at its shape.

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(population, bins=60, edgecolor="black")
plt.xlabel("Customer Spending")
plt.ylabel("Frequency")
plt.title("Population Distribution: Customer Spending")
plt.show()

The population is clearly not symmetric.

Now we will repeatedly take samples and calculate their means.

We will compare three sample sizes:

$$
n=5
$$

$$
n=30
$$

$$
n=100
$$

For each sample size, we will take 1,000 samples.

In [ ]:
def generate_sample_means(population, sample_size, repetitions=1000):
    means = []

    for i in range(repetitions):
        sample = np.random.choice(
            population,
            size=sample_size,
            replace=True
        )
        means.append(sample.mean())

    return np.array(means)


means_n5 = generate_sample_means(population, 5)
means_n30 = generate_sample_means(population, 30)
means_n100 = generate_sample_means(population, 100)

### Sample Size: $n=5$

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(means_n5, bins=30, edgecolor="black")
plt.axvline(population.mean(), linestyle="--", label="Population Mean")
plt.xlabel("Sample Mean")
plt.ylabel("Frequency")
plt.title("Sampling Distribution for n = 5")
plt.legend()
plt.show()

### Sample Size: $n=30$

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(means_n30, bins=30, edgecolor="black")
plt.axvline(population.mean(), linestyle="--", label="Population Mean")
plt.xlabel("Sample Mean")
plt.ylabel("Frequency")
plt.title("Sampling Distribution for n = 30")
plt.legend()
plt.show()

### Sample Size: $n=100$

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(means_n100, bins=30, edgecolor="black")
plt.axvline(population.mean(), linestyle="--", label="Population Mean")
plt.xlabel("Sample Mean")
plt.ylabel("Frequency")
plt.title("Sampling Distribution for n = 100")
plt.legend()
plt.show()

## 8. What Changed as Sample Size Increased?

Three things become visible.

### 1. The sampling distribution becomes more Normal

Even though the original customer spending population is skewed, the distribution of sample means becomes increasingly bell-shaped.

### 2. Sample means concentrate around the population mean

Larger samples tend to produce more stable estimates.

### 3. The spread of the sampling distribution decreases

The sample means vary less when the sample size increases.

This decreasing spread is explained by the Standard Error.

$$
SE = \frac{\sigma}{\sqrt{n}}
$$

In [ ]:
population_std = population.std()

se_5 = population_std / np.sqrt(5)
se_30 = population_std / np.sqrt(30)
se_100 = population_std / np.sqrt(100)

print("Standard Error for n = 5:", round(se_5, 2))
print("Standard Error for n = 30:", round(se_30, 2))
print("Standard Error for n = 100:", round(se_100, 2))

# 9. Standard Deviation vs Standard Error

These two measures answer different questions.

### Standard Deviation

Standard deviation describes the variability of individual observations.

$$
\sigma
=
\text{Spread of individual values in the population}
$$

### Standard Error

Standard error describes the variability of a sample statistic across repeated samples.

For the sample mean:

$$
SE = \frac{\sigma}{\sqrt{n}}
$$

A smaller Standard Error means that sample means are more tightly concentrated around the population mean.

As $n$ increases:

$$
\sqrt{n} \uparrow
$$

therefore:

$$
SE \downarrow
$$

This is one reason larger samples generally provide more precise estimates.

In [ ]:
sample_sizes = np.array([5, 10, 20, 30, 50, 100, 200, 500])
standard_errors = population_std / np.sqrt(sample_sizes)

plt.figure(figsize=(9, 4))
plt.plot(sample_sizes, standard_errors, marker="o")
plt.xlabel("Sample Size")
plt.ylabel("Standard Error")
plt.title("Effect of Sample Size on Standard Error")
plt.show()

### Practical Interpretation

Increasing sample size improves precision, but the improvement is not linear.

Because:

$$
SE = \frac{\sigma}{\sqrt{n}}
$$

reducing the Standard Error by half requires approximately four times the sample size.

This matters when deciding how much data to collect.

More data usually improves estimation, but collecting more data may also require additional time and cost.

# 10. Why the Central Limit Theorem Matters

The Central Limit Theorem creates a bridge between probability and inferential statistics.

Suppose we know only a sample.

We calculate:

$$
\bar{x}
$$

But we want to learn about:

$$
\mu
$$

The CLT tells us how sample means behave across repeated samples.

This allows us to quantify the uncertainty associated with our estimate.

The flow becomes:

$$
\text{Population}
\rightarrow
\text{Random Sample}
\rightarrow
\text{Sample Mean}
\rightarrow
\text{Sampling Distribution}
\rightarrow
\text{Inference}
$$

This idea is the foundation for:

- Confidence Intervals
- Hypothesis Testing
- Statistical Significance

The CLT does not directly make predictions about individual observations.

Instead, it helps us make reliable statements about population parameters using sample data.

# 11. Estimation Theory

Suppose a company wants to know the average amount spent by all its customers.

The population mean is:

$$
\mu
$$

but the company does not know its value.

A random sample is collected and the sample mean is calculated:

$$
\bar{x}
$$

Using a sample statistic to estimate an unknown population parameter is called **estimation**.

There are two common forms:

$$
\text{Point Estimation}
$$

and:

$$
\text{Interval Estimation}
$$

# 12. Point Estimation

A point estimate provides one value as an estimate of an unknown population parameter.

For example:

$$
\bar{x}
$$

can be used to estimate:

$$
\mu
$$

Similarly:

$$
\hat{p}
$$

can be used to estimate:

$$
p
$$

### Case Study: Average Customer Spending

Suppose we randomly select 100 customers.

Our business question is:

> What is our best estimate of the average spending of all customers?

In [ ]:
customer_sample = np.random.choice(
    population,
    size=100,
    replace=False
)

point_estimate = customer_sample.mean()

print("Sample Size:", len(customer_sample))
print("Estimated Average Customer Spending:", round(point_estimate, 2))

The sample mean is our point estimate:

$$
\hat{\mu} = \bar{x}
$$

This gives us one useful value.

However, another random sample would probably produce a slightly different estimate.

So the next question is:

> How much uncertainty is there around this estimate?

A single number cannot answer that.

For this, we use interval estimation.

# 13. Interval Estimation

An interval estimate gives a range of plausible values for an unknown population parameter.

Instead of reporting only:

$$
\bar{x}=2000
$$

we might report something like:

$$
1850 < \mu < 2150
$$

The most common interval estimate is the **Confidence Interval**.

A general form is:

$$
\text{Estimate}
\pm
\text{Margin of Error}
$$

For a population mean:

$$
\bar{x}
\pm
\text{Critical Value} \times \text{Standard Error}
$$

# 14. Confidence Interval

A confidence interval provides a range of values used to estimate an unknown population parameter.

Suppose we calculate a 95% confidence interval for average customer spending.

The interval might be:

$$
₹1800 \leq \mu \leq ₹2200
$$

The interval communicates more information than a point estimate because it also reflects uncertainty.

A wider interval indicates greater uncertainty.

A narrower interval indicates greater precision.

## 15. Confidence Level

Common confidence levels are:

$$
90\%
$$

$$
95\%
$$

$$
99\%
$$

A higher confidence level requires a wider interval because we want the method to capture the true parameter more often across repeated samples.

For a 95% confidence interval, the long-run interpretation is:

> If we repeatedly took random samples and constructed a confidence interval using the same method, approximately 95% of those intervals would contain the true population parameter.

The population parameter itself is fixed. The interval changes from sample to sample.

# 16. Confidence Interval for a Mean

When the population standard deviation is unknown, which is common in practice, we use the sample standard deviation and the $t$ distribution.

The confidence interval is:

$$
\bar{x}
\pm
t_{\alpha/2,n-1}
\left(
\frac{s}{\sqrt{n}}
\right)
$$

where:

- $\bar{x}$ = Sample mean
- $s$ = Sample standard deviation
- $n$ = Sample size
- $t_{\alpha/2,n-1}$ = Critical value from the $t$ distribution
- $\frac{s}{\sqrt{n}}$ = Estimated Standard Error

## 17. Case Study: Estimating Average Delivery Time

A delivery company wants to estimate its true average delivery time.

It cannot examine every future delivery, so it records a sample of recent deliveries.

The question is:

> Based on this sample, what range is likely to contain the true average delivery time?

In [ ]:
delivery_times = np.array([
    29, 31, 28, 34, 32, 30, 27, 35, 33, 31,
    29, 30, 32, 34, 28, 31, 33, 30, 29, 32,
    36, 27, 31, 30, 33, 29, 32, 34, 28, 31
])

n = len(delivery_times)
sample_mean = delivery_times.mean()
sample_std = delivery_times.std(ddof=1)

print("Sample Size:", n)
print("Sample Mean:", round(sample_mean, 2))
print("Sample Standard Deviation:", round(sample_std, 2))

First, calculate the Standard Error:

$$
SE = \frac{s}{\sqrt{n}}
$$

In [ ]:
standard_error = sample_std / np.sqrt(n)

print("Standard Error:", round(standard_error, 3))

Now calculate the 95% confidence interval.

For a 95% confidence level:

$$
\alpha = 1 - 0.95 = 0.05
$$

The remaining probability is divided between the two tails:

$$
\frac{\alpha}{2}=0.025
$$

In [ ]:
confidence_level = 0.95
alpha = 1 - confidence_level

t_critical = stats.t.ppf(
    1 - alpha / 2,
    df=n - 1
)

margin_of_error = t_critical * standard_error

lower_bound = sample_mean - margin_of_error
upper_bound = sample_mean + margin_of_error

print("T Critical Value:", round(t_critical, 3))
print("Margin of Error:", round(margin_of_error, 3))
print(
    "95% Confidence Interval:",
    (round(lower_bound, 2), round(upper_bound, 2))
)

### Interpretation

The sample mean gives the best single estimate of average delivery time.

The confidence interval gives a reasonable range of values for the unknown population mean.

The important distinction is:

$$
\text{Point Estimate}
=
\text{One estimated value}
$$

$$
\text{Confidence Interval}
=
\text{A range that reflects uncertainty}
$$

# 18. Margin of Error

The Margin of Error determines how far the confidence interval extends from the point estimate.

$$
\text{Margin of Error}
=
\text{Critical Value}
\times
\text{Standard Error}
$$

Therefore:

$$
CI
=
\bar{x}
\pm
ME
$$

The Margin of Error is affected by:

- Sample size
- Variability in the data
- Confidence level

In general:

### Larger Sample Size

$$
n \uparrow
\Rightarrow
SE \downarrow
\Rightarrow
ME \downarrow
$$

The interval becomes narrower.

### Greater Variability

$$
s \uparrow
\Rightarrow
SE \uparrow
\Rightarrow
ME \uparrow
$$

The interval becomes wider.

### Higher Confidence Level

Higher confidence requires a larger critical value.

Therefore, the interval becomes wider.

# 19. Case Study: How Sample Size Changes Precision

Suppose the standard deviation of customer spending is approximately ₹1,200.

Let us compare the Standard Error for different sample sizes.

In [ ]:
estimated_std = 1200

sample_sizes = [25, 100, 400]

for n in sample_sizes:
    se = estimated_std / np.sqrt(n)

    print(
        "Sample Size:",
        n,
        "| Standard Error:",
        round(se, 2)
    )

### Observation

As sample size increases, Standard Error decreases.

This means larger samples generally produce more precise estimates of the population mean.

However, notice the square-root relationship:

$$
SE \propto \frac{1}{\sqrt{n}}
$$

Increasing the sample size from 100 to 400 reduces the Standard Error by half.

This is useful when planning surveys, experiments, and data collection.

# 20. Case Study: How Confidence Level Changes the Interval

Using the same delivery-time sample, let us compare:

- 90% Confidence Interval
- 95% Confidence Interval
- 99% Confidence Interval

In [ ]:
confidence_levels = [0.90, 0.95, 0.99]

for confidence in confidence_levels:

    alpha = 1 - confidence

    t_critical = stats.t.ppf(
        1 - alpha / 2,
        df=n - 1
    )

    margin = t_critical * standard_error

    lower = sample_mean - margin
    upper = sample_mean + margin

    print(
        f"{int(confidence * 100)}% CI:",
        (round(lower, 2), round(upper, 2))
    )

### Observation

As the confidence level increases, the interval becomes wider.

This reflects a trade-off:

$$
\text{Higher Confidence}
\Longrightarrow
\text{Wider Interval}
$$

$$
\text{Narrower Interval}
\Longrightarrow
\text{Lower Confidence, all else equal}
$$

There is no universally correct confidence level.

A 95% confidence level is commonly used, but the choice depends on the context and the consequences of uncertainty.

# 21. From Confidence Intervals to Business Decisions

Suppose a delivery company promises:

$$
\text{Average Delivery Time} = 30 \text{ minutes}
$$

Our sample produces an estimated average and a confidence interval.

Now a new question appears:

> Is the true average delivery time still consistent with the company's 30-minute claim?

This is no longer only an estimation problem.

It is a decision problem.

We want to compare:

$$
\text{Observed Sample Evidence}
$$

against:

$$
\text{A Claimed Population Value}
$$

This is where hypothesis testing begins.

In [ ]:
claimed_mean = 30

print("Sample Mean:", round(sample_mean, 2))
print(
    "95% Confidence Interval:",
    (round(lower_bound, 2), round(upper_bound, 2))
)
print("Company Claim:", claimed_mean, "minutes")

## 22. The Connection to Hypothesis Testing

Estimation asks:

> What is the likely value of the population parameter?

Hypothesis testing asks:

> Is there enough evidence in the sample to challenge a specific claim about the population?

For example:

### Estimation Question

> What is the average delivery time?

We might answer with:

$$
\bar{x}
$$

and a confidence interval.

### Hypothesis Testing Question

> Is the average delivery time different from 30 minutes?

Now we begin with a claim:

$$
H_0:\mu=30
$$

and use sample evidence to decide whether that claim remains reasonable.

The ideas developed in this module form the foundation:

$$
\text{Sampling}
\rightarrow
\text{Sampling Distribution}
\rightarrow
\text{CLT}
\rightarrow
\text{Standard Error}
\rightarrow
\text{Estimation}
\rightarrow
\text{Confidence Interval}
\rightarrow
\text{Hypothesis Testing}
$$

# 23. Quick Practice

## Problem 1: Customer Spending

A sample of 100 customers has:

$$
\bar{x}=₹2500
$$

and:

$$
s=₹800
$$

Calculate the Standard Error.

Use:

$$
SE=\frac{s}{\sqrt{n}}
$$

---

## Problem 2: Sample Size

Two analysts estimate average customer spending.

Analyst A uses 25 customers.

Analyst B uses 400 customers.

Assuming both samples come from the same population, which estimate is expected to have a smaller Standard Error?

Explain why.

---

## Problem 3: Point vs Interval Estimate

A company reports:

$$
\text{Average estimated delivery time}=31.2\text{ minutes}
$$

Is this a point estimate or an interval estimate?

What additional information would an interval estimate provide?

---

## Problem 4: Confidence Level

Suppose a 95% confidence interval is:

$$
28.5 < \mu < 32.5
$$

Would a 99% confidence interval generally be narrower or wider?

Why?

---

## Problem 5: From Estimation to Testing

A company claims that its average delivery time is 25 minutes.

A sample suggests an average delivery time of 31 minutes.

Can we immediately conclude that the company's claim is false?

What statistical question should we ask before making that decision?

# Module Summary

This module moved from probability distributions to statistical inference.

The complete process is:

$$
\boxed{
\text{Population}
\rightarrow
\text{Sample}
\rightarrow
\text{Sampling Distribution}
\rightarrow
\text{Central Limit Theorem}
\rightarrow
\text{Estimation}
}
$$

The main ideas are:

| Concept | Main Idea |
|---|---|
| Population | Complete group we want to understand |
| Sample | Subset of the population we observe |
| Parameter | Numerical characteristic of a population |
| Statistic | Numerical characteristic calculated from a sample |
| Sampling Distribution | Distribution of a statistic across repeated samples |
| Central Limit Theorem | Sample means tend toward a Normal distribution as sample size becomes sufficiently large |
| Standard Error | Measures the variability of a sample statistic |
| Point Estimation | Uses one value to estimate a population parameter |
| Interval Estimation | Uses a range to estimate a population parameter |
| Confidence Interval | Range of plausible values for a population parameter |
| Margin of Error | Determines the width around the point estimate |
| Confidence Level | Long-run success rate of the interval-building procedure |

The central idea is simple:

> A sample gives us information, but it also contains uncertainty.

Inferential statistics gives us a structured way to measure that uncertainty.

The next question is:

> When a sample result differs from a claimed or expected value, is the difference large enough to be considered statistically significant?

That question leads directly to **Hypothesis Testing**.